This is a Jupyter notebook for analyzing the Olist Brazilian E-commerce dataset.
Practicing in python with pandas and SQL to prepare for Elevate technical assessment. Working with Claude Code and visualizing in Windsurf.

<img src="olist_schema.png" width="700"/>

The Fictional Brief:
Elevate Portfolio Analytics — Olist Integration Project
Olist is a Brazilian e-commerce marketplace that connects small merchants to major retail channels. Your client, a PE firm, has acquired three regional merchant clusters operating on Olist's platform. Each cluster has its own sellers, customers, and order history. Your job is to unify and analyze the combined dataset to surface operational insights and flag data quality issues — exactly the kind of integration and BI work Elevate does across its roll-up companies.

## Phase 1 — Load, Inspect & Map Schema

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = Path(".")

In [ ]:
tables = {
    "orders":        pd.read_csv(DATA_DIR / "olist_orders_dataset.csv"),
    "order_items":   pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv"),
    "order_reviews": pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv"),
    "customers":     pd.read_csv(DATA_DIR / "olist_customers_dataset.csv"),
    "payments":      pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv"),
    "products":      pd.read_csv(DATA_DIR / "olist_products_dataset.csv"),
    "sellers":       pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv"),
    "geo":           pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv"),
    "category_xlat": pd.read_csv(DATA_DIR / "product_category_name_translation.csv"),
}
print("Loaded tables:", list(tables.keys()))

### Shape & Dtype Inspection

For each table: row/column counts, column dtypes, null counts, null %, and cardinality.

In [ ]:
for name, df in tables.items():
    print(f"\n{'='*55}")
    print(f"TABLE: {name}  —  {df.shape[0]:,} rows × {df.shape[1]} cols")
    summary = pd.DataFrame({
        "dtype":    df.dtypes,
        "nulls":    df.isnull().sum(),
        "null_%":   (df.isnull().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    display(summary)

### Schema Relationship Map

```
orders (order_id PK, customer_id FK → customers.customer_id)
  ├─ order_items   (order_id FK, product_id FK → products, seller_id FK → sellers)
  ├─ payments      (order_id FK)
  └─ order_reviews (order_id FK)

customers    (customer_id PK, customer_zip_code_prefix → geo)
sellers      (seller_id PK,   seller_zip_code_prefix   → geo)
products     (product_id PK,  product_category_name FK → category_xlat)
geo          (geolocation_zip_code_prefix — NOT a unique PK; multiple lat/lng per zip)
category_xlat(product_category_name PK,  product_category_name_english)
```

**Cluster key** — `sellers.seller_state`:

| State | Cluster   |
|-------|-----------|
| SP    | Cluster A |
| RJ    | Cluster B |
| MG    | Cluster C |

**FK → PK reference table**

| Child table   | FK column                   | Parent table  | PK column                          |
|---------------|-----------------------------|---------------|------------------------------------|
| orders        | customer_id                 | customers     | customer_id                        |
| order_items   | order_id                    | orders        | order_id                           |
| order_items   | seller_id                   | sellers       | seller_id                          |
| order_items   | product_id                  | products      | product_id                         |
| payments      | order_id                    | orders        | order_id                           |
| order_reviews | order_id                    | orders        | order_id                           |
| products      | product_category_name       | category_xlat | product_category_name              |
| customers     | customer_zip_code_prefix    | geo           | geolocation_zip_code_prefix (non-unique) |
| sellers       | seller_zip_code_prefix      | geo           | geolocation_zip_code_prefix (non-unique) |

### Referential Integrity Checks

Verify FK → PK joins across all core relationships. Large orphan counts flag data quality gaps worth investigating in later phases.

In [ ]:
checks = {
    "order_items.order_id   → orders":    (tables["order_items"]["order_id"],
                                            tables["orders"]["order_id"]),
    "order_items.seller_id  → sellers":   (tables["order_items"]["seller_id"],
                                            tables["sellers"]["seller_id"]),
    "order_items.product_id → products":  (tables["order_items"]["product_id"],
                                            tables["products"]["product_id"]),
    "orders.customer_id     → customers": (tables["orders"]["customer_id"],
                                            tables["customers"]["customer_id"]),
    "payments.order_id      → orders":    (tables["payments"]["order_id"],
                                            tables["orders"]["order_id"]),
    "reviews.order_id       → orders":    (tables["order_reviews"]["order_id"],
                                            tables["orders"]["order_id"]),
    "products.category      → xlat":      (tables["products"]["product_category_name"].dropna(),
                                            tables["category_xlat"]["product_category_name"]),
}

print(f"{'Relationship':<45}  {'Orphans':>10}  {'Total FK':>10}  {'Orphan %':>9}")
print("-" * 80)
for label, (fk_col, pk_col) in checks.items():
    orphans = (~fk_col.isin(pk_col)).sum()
    total   = len(fk_col)
    pct     = orphans / total * 100 if total else 0
    flag    = " ⚠️" if orphans > 0 else ""
    print(f"{label:<45}  {orphans:>10,}  {total:>10,}  {pct:>8.2f}%{flag}")

### Cluster Assignment

Tag each seller by state → cluster label. This is the segmentation key for all downstream analysis.

In [ ]:
CLUSTER_MAP = {"SP": "Cluster A", "RJ": "Cluster B", "MG": "Cluster C"}

sellers = tables["sellers"].copy()
sellers["cluster"] = sellers["seller_state"].map(CLUSTER_MAP).fillna("Other")
tables["sellers"] = sellers  # update shared dict for downstream use

cluster_counts = sellers["cluster"].value_counts().rename_axis("cluster").reset_index(name="seller_count")
cluster_counts["pct"] = (cluster_counts["seller_count"] / cluster_counts["seller_count"].sum() * 100).round(1)
display(cluster_counts)